# 03f — Pre-Flight Risk Model (Problem C)

Train, calibrate, and evaluate the LightGBM pre-flight risk classifier.

**Input:** `data/processed/preflight_features_final.parquet`  
**Output:** Trained model + evaluation metrics

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
os.chdir(root)

src_path = str(root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Working directory: {root}')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

IN_PATH = Path('data/processed/preflight_features_final.parquet')
df = pd.read_parquet(IN_PATH)
print(f'Dataset: {len(df):,} rows')
print(f'Incident rate: {df["incident"].mean()*100:.2f}%')

# Compute class imbalance ratio for scale_pos_weight
n_neg = (df['incident'] == 0).sum()
n_pos = (df['incident'] == 1).sum()
scale_pos = n_neg / n_pos
print(f'Class ratio (neg/pos): {scale_pos:.1f}x')

# Feature set — only include columns actually present in the data
_candidate_features = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour',
    'temp', 'rhum', 'prcp', 'wspd', 'pres',
    'airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate',
    'DISTANCE'
]
features = [f for f in _candidate_features if f in df.columns]
target   = 'incident'
print(f'Features ({len(features)}): {features}')

## 1. Temporal Train/Val/Test Split

In [ ]:
df['year'] = pd.to_datetime(df['FL_DATE']).dt.year

train_df = df[df['year'] == 2018]
val_df   = df[df['year'] == 2019]
test_df  = df[df['year'] == 2020]

X_train, y_train = train_df[features], train_df[target]
X_val,   y_val   = val_df[features],   val_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

print(f'Train (2018): {len(X_train):,} rows | incidents: {y_train.sum():,}')
print(f'Val   (2019): {len(X_val):,} rows | incidents: {y_val.sum():,}')
print(f'Test  (2020): {len(X_test):,} rows | incidents: {y_test.sum():,}')

## 2. Train LightGBM with Early Stopping

In [ ]:
# scale_pos_weight: explicitly control class weight instead of is_unbalance
# This avoids the early stopping trigger at iteration=1 seen with is_unbalance on large datasets
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    scale_pos_weight=scale_pos,  # explicit imbalance handling
    random_state=42,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

n_iter = getattr(model, 'best_iteration_', model.n_estimators)
print(f'Best iteration: {n_iter}')
print('Model training complete.')

## 3. Probability Calibration

In [ ]:
# FrozenEstimator: sklearn 1.6+ recommended way to use prefit estimator in CalibratedClassifierCV
calibrated_model = CalibratedClassifierCV(FrozenEstimator(model), method='isotonic')
calibrated_model.fit(X_val, y_val)

fig, ax = plt.subplots(figsize=(7, 7))
CalibrationDisplay.from_estimator(calibrated_model, X_test, y_test, n_bins=10, ax=ax, name='LightGBM')
ax.set_title('Calibration Plot (Test Set 2020)')
plt.tight_layout()
plt.show()

## 4. Evaluation

In [ ]:
y_prob = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

roc_auc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)

print(f'ROC-AUC:           {roc_auc:.4f}')
print(f'Average Precision: {avg_prec:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Incident'], zero_division=0))

## 5. Feature Importance (SHAP)

In [ ]:
import shap

# Use the underlying lgb model for SHAP (not the calibrated wrapper)
sample_X = X_test.sample(min(1000, len(X_test)), random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample_X)

# For binary LightGBM, shap_values is a list [neg_class, pos_class]
# Use index 1 (positive/incident class)
sv_pos = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(sv_pos, sample_X, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Incident Class)')
plt.tight_layout()
plt.show()

## 6. Save Model Artifact

In [ ]:
model_dir = Path('models')
model_dir.mkdir(exist_ok=True)

model_path = model_dir / 'preflight_lgbm_calibrated.joblib'
joblib.dump({
    'model': calibrated_model,
    'features': features,
    'metrics': {
        'roc_auc': roc_auc,
        'avg_precision': avg_prec,
    }
}, model_path)

print(f'Model saved to {model_path}')
print(f'  ROC-AUC: {roc_auc:.4f}')
print(f'  Avg Precision: {avg_prec:.4f}')